## <a name="0">Fine-tuning an LLM on Amazon SageMaker</a>

In this solution, you explore how to fine-tune a pre-trained large languages model (LLM), which is a powerful technique in the field of generative artificial intelligence (AI). LLMs are pre-trained on enormous amounts of data, making them highly effective in grasping the nuances of language and generating coherent responses. These models have learned to extract useful features and patterns from the data, making them a valuable resource for various machine learning (ML) tasks.

With fine-tuning, also known as transfer learning, we can use the knowledge gained by a pre-trained model and apply it to a different but related task. Instead of training a model from scratch, we start with a pre-trained model and modify it to adapt to our specific problem domain. This approach saves significant computational resources, and it benefits from the generalization capabilities of the pre-trained model.

This notebook walks you through the step-by-step process of fine-tuning a pre-trained model. It covers the following primary steps:

1. <a href="#step1">Check GPU memory</a>
2. <a href="#step2">Import libraries</a>
3. <a href="#step3">Prepare the training dataset</a>
4. <a href="#step4">Load a pre-trained LLM</a>
5. <a href="#step5">Define the trainer and fine-tune the LLM</a>
6. <a href="#step6">Deploy the fine-tuned model</a>
7. <a href="#step7">Test the deployed inference</a>


NOTE: Work from the top to the bottom of this notebook, and do not skip sections; otherwise, you might receive error messages from missing code.


## <a name="step1">Step 1: Check GPU memory</a>

To check the GPU memory, run the following command.

In [ ]:
!nvidia-smi

# CHECK GPU MEMORY AVAILABILITY
# Why: Ensures sufficient GPU memory is available for model training. 
# If memory is insufficient, other notebooks must be closed to avoid out-of-memory errors.

If your CUDA memory is occupied by more than half (as in the following image), you need to shut down other running notebooks.

<p style="padding: 10px; border: 1px solid black;">
<img src="images/memory.png" alt="drawing" width="800"/> <br/>

## <a name="step2">Step 2: Import libraries</a>

Run the following two code blocks sequentially, one at a time, to import the necessary libraries, including the Hugging Face Transformers library and the PyTorch library, which is a dependency for Transformers.


In [ ]:
%%capture
!pip install -r requirements.txt

# INSTALL REQUIRED DEPENDENCIES
# Why: Installs all necessary Python packages listed in requirements.txt 
# (PyTorch, Transformers, PEFT, etc.) needed for fine-tuning workflow.

In [ ]:
%%capture
import warnings
warnings.filterwarnings("ignore")  # Suppress all warnings for cleaner output
import os
import numpy as np
import pandas as pd
from typing import Any, Dict, List, Tuple, Union
from datasets import Dataset, load_dataset, disable_caching
disable_caching() ## Disable huggingface dataset caching to save memory
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer
from transformers import TextDataset
import torch
from torch.utils.data import Dataset, random_split
from transformers import TrainingArguments, Trainer
import accelerate
import bitsandbytes
from IPython.display import Markdown

# IMPORT ALL NECESSARY LIBRARIES FOR LLM FINE-TUNING
# Why: Load essential libraries for data processing, model loading, training, and tokenization.
# - warnings: Suppress non-critical warnings to reduce console noise
# - pandas/numpy: Data manipulation and numerical operations
# - datasets: Hugging Face library for efficient dataset handling
# - transformers: Pre-trained models and tokenizers from Hugging Face
# - torch: PyTorch deep learning framework
# - accelerate/bitsandbytes: Advanced training optimization and quantization
# - IPython: Display rich output in notebooks

## <a name="step3">Step 3: Prepare the training dataset</a>

Load and view the dataset. For this practice lab, you use [Amazon SageMaker FAQs](https://aws.amazon.com/sagemaker/faqs/) for the main dataset, which has two columns: `instruction` and `response`.

In [ ]:
sagemaker_faqs_dataset = load_dataset("csv",
                                      data_files='data/amazon_sagemaker_faqs.csv')['train']
sagemaker_faqs_dataset

# LOAD TRAINING DATASET FROM CSV FILE
# Why: Load Amazon SageMaker FAQ data which contains instruction-response pairs.
# Dataset structure: 'instruction' column (questions) and 'response' column (answers)
# Used for fine-tuning the LLM to answer domain-specific questions.

In [ ]:
sagemaker_faqs_dataset[0]

# DISPLAY FIRST DATA SAMPLE
# Why: Verify the structure and content of a single dataset record 
# to understand instruction-response format before preprocessing.

### <a name="step3">Step 3.1: Prepare the prompt</a>
To fine-tune the LLM, you must decorate the instruction dataset with a PROMPT, as shown below.

In [ ]:
from utils.helpers import INTRO_BLURB, INSTRUCTION_KEY, RESPONSE_KEY, END_KEY, RESPONSE_KEY_NL, DEFAULT_SEED, PROMPT
'''
PROMPT = """{intro}
            {instruction_key}
            {instruction}
            {response_key}
            {response}
            {end_key}"""
'''
Markdown(PROMPT)

# IMPORT AND DISPLAY PROMPT TEMPLATE
# Why: Load the predefined prompt template structure from helpers module.
# The template formats instruction-response pairs into a standardized prompt format
# for the model to learn the correct response pattern during fine-tuning.

Now, feed the PROMPT to the dataset through the `_add_text` Python function, which takes a record as input. The function checks that both fields (instruction/response) are not null, and then passes the values to the predefined PROMPT template.

In [ ]:
def _add_text(rec):
    instruction = rec["instruction"]
    response = rec["response"]
    if not instruction:
        raise ValueError(f"Expected an instruction in: {rec}")
    if not response:
        raise ValueError(f"Expected a response in: {rec}")
    rec["text"] = PROMPT.format(
        instruction=instruction, response=response)
    return rec

# PREPARE DATASET RECORDS WITH FORMATTED PROMPT TEMPLATE
# Why: Transform raw instruction-response pairs into a structured text format
# that the LLM will learn from. Validates non-null fields and applies the prompt template
# to each record for consistent formatting during fine-tuning.
    # Validate that required fields exist
    # Apply prompt template to combine instruction and response

In [ ]:
sagemaker_faqs_dataset = sagemaker_faqs_dataset.map(_add_text)
sagemaker_faqs_dataset[0]

# APPLY PROMPT TEMPLATE TO ENTIRE DATASET
# Why: Map the _add_text function across all dataset records to create 
# a 'text' field with formatted prompts for each instruction-response pair.

Use `Markdown` to neatly display the text.

In [ ]:
Markdown(sagemaker_faqs_dataset[0]['text'])

# DISPLAY FORMATTED PROMPT IN MARKDOWN
# Why: Render the first processed record's formatted text in readable markdown format
# to visually verify that the prompt template was applied correctly.

## <a name="#step4">Step 4: Load a pre-trained LLM</a>


To load a pre-trained model, initialize a tokenizer and a base model by using the `databricks/dolly-v2-3b` model from the Hugging Face Transformers library. The tokenizer converts raw text into tokens, and the base model generates text based on a given prompt. By following the instructions previously outlined, you can correctly instantiate these components and use their functionality in your code.


The `AutoTokenizer.from_pretrained()` Python function is used to instantiate the tokenizer.
- `padding_side="left"` specifies the side of the sequences where padding tokens are added. In this case, padding tokens are added to the left side of each sequence.
- `eos_token` is a special token that represents the end of a sequence. By assigning the token to `pad_token`, any padding tokens added during tokenization are also considered end-of-sequence tokens. This can be useful when generating text through the model because the model knows when to stop generating text after encountering padding tokens.
- `tokenizer.add_special_tokens...` adds three additional special tokens to the tokenizer's vocabulary. These tokens likely serve specific purposes in the application using the tokenizer. For example, the tokens could be used to mark the end of an input, an instruction, or a response in a dialogue system.

After the function runs, the `tokenizer` object has been initialized and is ready to use for tokenizing text.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("verseAI/databricks-dolly-v2-3b",
                                          padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.add_special_tokens({"additional_special_tokens":
                              [END_KEY, INSTRUCTION_KEY, RESPONSE_KEY_NL]})

# INITIALIZE TOKENIZER FOR MODEL PREPROCESSING
# Why: Create a tokenizer that converts raw text into token IDs that the model understands.
# - padding_side="left": Pad sequences on the left to handle variable-length inputs
# - pad_token = eos_token: Treat padding as end-of-sequence to signal model when to stop
# - add_special_tokens: Register custom tokens (END_KEY, INSTRUCTION_KEY, RESPONSE_KEY_NL) 
#   for proper instruction-response structure recognition during training

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "verseAI/databricks-dolly-v2-3b",
    device_map="auto", 
    load_in_8bit=True,  # Reduces memory footprint without significant accuracy loss
)

# LOAD PRE-TRAINED MODEL WITH 8-BIT QUANTIZATION
# Why: Load a pre-trained Dolly model optimized for resource efficiency:
# - device_map="auto": Automatically distribute model across available GPUs
# - load_in_8bit=True: Apply 8-bit quantization to reduce memory usage by ~4x
#   while maintaining model performance (critical for large models on limited GPU memory)

### <a name="#step4.1">Step 4.1: Prepare the model for training</a>
Some preprocessing is needed before training an INT8 model using Parameter-Efficient Fine-Tuning (PEFT); therefore, import a utility function, `prepare_model_for_int8_training`, that will:

- Cast all the non-INT8 modules to full precision (FP32) for stability.
- Add a forward_hook to the input embedding layer to enable gradient computation of the input hidden states.
- Enable gradient checkpointing for more memory-efficient training.

In [ ]:
model.resize_token_embeddings(len(tokenizer))

# RESIZE MODEL TOKEN EMBEDDINGS
# Why: Expand the model's token embedding layer to accommodate new special tokens
# added to the tokenizer (END_KEY, INSTRUCTION_KEY, RESPONSE_KEY_NL).
# This ensures the model can properly process these custom tokens during training.

Use the `preprocess_batch` function to preprocess the text field of the batch, applying tokenization, truncation, and other relevant operations based on the specified maximum length. The field takes a batch of data, a tokenizer, and a maximum length as input.

For more details, refer to `mlu_utils/helpers.py` file.

In [ ]:
from functools import partial
from utils.helpers import mlu_preprocess_batch
MAX_LENGTH = 256  # Maximum token sequence length
_preprocessing_function = partial(mlu_preprocess_batch, max_length=MAX_LENGTH, tokenizer=tokenizer)

# DEFINE PREPROCESSING FUNCTION WITH TOKENIZATION
# Why: Create a partial function that tokenizes, truncates, and pads text to MAX_LENGTH.
# - MAX_LENGTH=256: Limits sequence length to fit model's context window efficiently
# - partial(): Pre-configures preprocessing function with tokenizer and max_length
# This function will be applied to all dataset batches before training.

Next, apply the preprocessing function to each batch in the dataset, modifying the text field accordingly. The map operation is performed in a batched manner and the instruction, response, and text columns are removed from the dataset. Finally, `processed_dataset` is created by filtering `sagemaker_faqs_dataset` based on the length of the input_ids field, ensuring that it fits within the specified `MAX_LENGTH`.

In [ ]:
encoded_sagemaker_faqs_dataset = sagemaker_faqs_dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=["instruction", "response", "text"],
)
processed_dataset = encoded_sagemaker_faqs_dataset.filter(lambda rec: len(rec["input_ids"]) < MAX_LENGTH)

# TOKENIZE DATASET AND FILTER BY SEQUENCE LENGTH
# Why: Apply tokenization preprocessing to all dataset batches:
# - batched=True: Process multiple samples at once for efficiency
# - remove_columns: Drop original text columns to save memory
# - filter(): Remove sequences exceeding MAX_LENGTH to prevent training errors
# Result: A dataset with 'input_ids' and 'attention_mask' ready for model training

Split the dataset into `train` and `test` for evaluation.

In [ ]:
split_dataset = processed_dataset.train_test_split(test_size=14, seed=0)
split_dataset

# SPLIT DATASET INTO TRAINING AND EVALUATION SETS
# Why: Create separate train (90%) and test (10%) splits for model evaluation:
# - test_size=14: Reserve 14 samples for validation testing
# - seed=0: Use fixed seed for reproducible splits across runs
# This allows monitoring model generalization on unseen data during training.

## <a name="#step5">Step 5: Define the trainer and fine-tune the LLM</a>

To efficiently fine-tune a model, in this practice lab, you use [LoRA: Low-Rank Adaptation](https://arxiv.org/abs/2106.09685). LoRA freezes the pre-trained model weights and injects trainable rank decomposition matrices into each layer of the Transformer architecture, greatly reducing the number of trainable parameters for downstream tasks. Compared to GPT-3 175B fine-tuned with Adam, LoRA can reduce the number of trainable parameters by 10,000 times and reduce the GPU memory requirement by 3 times.


### <a name="#step5.1">Step 5.1: Define LoraConfig and load the LoRA model</a>

Use the build LoRA class `LoraConfig` from [huggingface 🤗 PEFT: State-of-the-art Parameter-Efficient Fine-Tuning](https://github.com/huggingface/peft). Within `LoraConfig`, specify the following parameters:

- `r`, the dimension of the low-rank matrices
- `lora_alpha`, the scaling factor for the low-rank matrices
- `lora_dropout`, the dropout probability of the LoRA layers

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_int8_training, TaskType
model = prepare_model_for_int8_training(model)
for param in model.parameters():
    if param.dtype == torch.float32 or param.dtype == torch.float16:
        param.requires_grad = False
num_layers_to_unfreeze = 2
for i, layer in enumerate(model.gpt_neox.layers[-num_layers_to_unfreeze:]):
    for param in layer.parameters():
        if param.dtype == torch.float32 or param.dtype == torch.float16:
            param.requires_grad = True
MICRO_BATCH_SIZE = 8
BATCH_SIZE = 64
GRADIENT_ACCUMULATION_STEPS = BATCH_SIZE // MICRO_BATCH_SIZE  # Effective batch size simulation
LORA_R = 256  # Low-rank decomposition dimension
LORA_ALPHA = 512  # Scaling factor for LoRA weights
LORA_DROPOUT = 0.05  # Dropout rate in LoRA layers
lora_config = LoraConfig(
                 r=LORA_R,
                 lora_alpha=LORA_ALPHA,
                 lora_dropout=LORA_DROPOUT,
                 bias="none",
                 task_type="CAUSAL_LM"  # Specify causal language modeling task
)

# CONFIGURE AND APPLY LORA ADAPTER FOR PARAMETER-EFFICIENT FINE-TUNING
# Why: Use LoRA (Low-Rank Adaptation) to reduce trainable parameters by 10,000x:
# 1. Prepare model for 8-bit training (enables mixed precision support)
# 2. Freeze all base model parameters (prevents catastrophic forgetting)
# 3. Unfreeze only top 2 layers (allows task-specific adaptation)
# 4. LoRA creates small adapter matrices (r=256, alpha=512) instead of fine-tuning entire model
#    This drastically reduces GPU memory and training time while maintaining performance.
# Prepare 8-bit model for training with proper gradient computation setup
# Freeze all parameters initially
# Unfreeze only top 2 layers for adaptation (preserves pre-trained knowledge in lower layers)
# Training hyperparameters
# Define LoRA configuration

Use the `get_peft_model` function to initialize the model with the LoRA framework, configuring it based on the provided `lora_config` settings. This way, the model can incorporate the benefits and capabilities of the LoRA optimization approach.

In [ ]:
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# WRAP MODEL WITH LORA ADAPTER AND DISPLAY TRAINABLE PARAMETERS
# Why: Apply LoRA configuration to the model, reducing trainable parameters from millions to thousands.
# print_trainable_parameters() shows the efficiency gain (typically ~3% of original model size).
# This dramatic reduction enables training on consumer-grade GPUs without full fine-tuning overhead.

As shown, LoRA-only trainable parameters are about 3 percent of the full weights, which is much more efficient.

### <a name="#step5.2">Step 5.2: Define the data collator</a>

A DataCollator is a huggingface🤗 transformers function that takes a list of samples from a dataset and collates them into a batch, as a dictionary of PyTorch tensors.

Use `DataCollatorForCompletionOnlyLM`, which extends the functionality of the base `DataCollatorForLanguageModeling` class from the Transformers library. This custom collator is designed to handle examples where a prompt is followed by a response in the input text and the labels are modified accordingly.

For implementation, refer to `utils/helpers.py`.

In [ ]:
from utils.helpers import MLUDataCollatorForCompletionOnlyLM
data_collator = MLUDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer, mlm=False, return_tensors="pt", pad_to_multiple_of=8
)

# INITIALIZE DATA COLLATOR FOR BATCH PREPROCESSING
# Why: Create a custom data collator that:
# - Handles instruction-response pairs intelligently
# - Only computes loss on response tokens (ignores instruction tokens)
# - Applies dynamic padding to variable-length sequences
# - Pads to multiple of 8 for hardware efficiency
# This ensures the model learns to generate correct responses, not just repeat instructions.

### <a name="#step5.3">Step 5.3: Define the trainer</a>

To fine-tune the LLM, you must define a trainer. First, define the training arguments.

In [ ]:
EPOCHS = 10
LEARNING_RATE = 1e-4
MODEL_SAVE_FOLDER_NAME = "dolly-3b-lora"
training_args = TrainingArguments(
                    output_dir=MODEL_SAVE_FOLDER_NAME,
                    fp16=True,  # Enable half-precision for efficiency
                    per_device_train_batch_size=8,
                    per_device_eval_batch_size=8,
                    learning_rate=LEARNING_RATE,
                    num_train_epochs=EPOCHS,
                    logging_strategy="steps",
                    logging_steps=100,  # Log metrics every 100 steps
                    evaluation_strategy="steps",
                    eval_steps=100,  # Evaluate every 100 steps
                    save_strategy="steps",
                    save_steps=20000,  # Save checkpoint every 20000 steps
                    save_total_limit=10,  # Keep only 10 most recent checkpoints
                    optim="adamw_torch"
)

# CONFIGURE TRAINING HYPERPARAMETERS AND STRATEGY
# Why: Define the training process configuration:
# - EPOCHS=10: Train for 10 passes through the dataset for sufficient convergence
# - LEARNING_RATE=1e-4: Conservative learning rate for fine-tuning (lower than from-scratch training)
# - fp16=True: Use 16-bit floating point for 2x memory savings and faster computation
# - per_device_batch_size=8: Batch size for each GPU
# - eval_steps=100: Evaluate on test set every 100 steps to monitor overfitting
# - save_steps=20000: Save model checkpoints at intervals for recovery
# - adamw_torch: Stable optimizer variant with weight decay regularization

This is where the magic happens! Initialize the trainer with the defined model, tokenizer, training arguments, data collator, and the train/eval datasets.

The training takes about 10 minutes.

In [ ]:
trainer = Trainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        train_dataset=split_dataset['train'],
        eval_dataset=split_dataset["test"],
        data_collator=data_collator,
)
model.config.use_cache = False  # Disable cache during training to save memory
trainer.train()  # Start fine-tuning

# INITIALIZE TRAINER AND START FINE-TUNING PROCESS
# Why: Create a trainer object that orchestrates the entire training loop:
# - Combines model, tokenizer, training arguments, and datasets
# - Handles forward/backward passes, gradient updates, and validation
# - use_cache=False: Disable KV-cache during training (saves memory, enabled later for inference)
# - trainer.train(): Execute the fine-tuning process for all epochs
# Typically takes 10-15 minutes on GPU depending on dataset size.

### <a name="#step5.4">Step 5.4: Save the fine-tuned model</a>


When the training is completed, you can save the model to a directory by using the [`transformers.PreTrainedModel.save_pretrained`] function.
This function saves only the incremental 🤗 PEFT weights (adapter_model.bin) that were trained, so the model is very efficient to store, transfer, and load.

In [ ]:
trainer.model.save_pretrained(MODEL_SAVE_FOLDER_NAME)

# SAVE PEFT ADAPTER WEIGHTS TO DISK
# Why: Save only the LoRA adapter weights (adapter_model.bin) instead of full model.
# Advantages:
# - Adapter file is ~100MB vs ~7GB for full model (70x smaller)
# - Faster to download and transfer
# - Can be combined with base model for inference
# - Efficient for model deployment and sharing

If you want to save the full model that you just fine-tuned, you can use the [`transformers.trainer.save_model`] function. Meanwhile, save the training arguments together with the trained model.

In [ ]:
trainer.save_model()

# SAVE TRAINER STATE AND TRAINING CONFIGURATION
# Why: Preserve complete training state including:
# - Training arguments and hyperparameters
# - Optimizer state
# - Training logs and metrics
# Useful for resuming training or analyzing training history.

In [ ]:
trainer.model.config.save_pretrained(MODEL_SAVE_FOLDER_NAME)

# SAVE MODEL CONFIGURATION
# Why: Persist the model's configuration settings including:
# - Architecture details (number of layers, hidden dimensions, etc.)
# - Attention mechanisms
# - Any customizations made during fine-tuning
# Essential for loading and using the model later without errors.

Save the tokenizer along with the trained model.

In [ ]:
tokenizer.save_pretrained(MODEL_SAVE_FOLDER_NAME)

# SAVE TOKENIZER FOR INFERENCE
# Why: Persist the tokenizer with special tokens added during training.
# During inference, the same tokenizer must be used to ensure:
# - Consistent token encoding/decoding
# - Recognition of custom special tokens (END_KEY, INSTRUCTION_KEY, RESPONSE_KEY_NL)
# - Identical preprocessing to training data

## <a name="#step6">Step 6: Deploy the fine-tuned model</a>

### <a name="step6">Overview of deployment parameters</a>

To deploy using the Amazon SageMaker Python SDK with the Deep Java Library (DJL), you must instantiate the `Model` class with the following parameters:
```{python}
model = Model(
    image_uri,
    model_data=...,
    predictor_cls=...,
    role=aws_role
)
```
- `image_uri`: The Docker image URI representing the deep learning framework and version to be used.
- `model_data`: The location of the fine-tuned LLM model artifact in an Amazon Simple Storage Service (Amazon S3) bucket. It specifies the path to the TAR GZ file containing the model's parameters, architecture, and any necessary artifacts.
- `predictor_cls`: This is just a JSON in JSON out predictor, nothing DJL related. For more information, see [sagemaker.djl_inference.DJLPredictor](https://sagemaker.readthedocs.io/en/stable/frameworks/djl/sagemaker.djl_inference.html#djlpredictor).
- `role`: The AWS Identity and Access Management (IAM) role ARN that provides necessary permissions to access resources, such as the S3 bucket that contains the model data.

### <a name="step6.1">Step 6.1: Instantiate SageMaker parameters</a>

Initialize an Amazon SageMaker session and retrieve information related to the AWS environment such as the SageMaker role and AWS Region. You also specify the image URI for a specific version of the "djl-deepspeed" framework by using the SageMaker session's Region. The image URI is a unique identifier for a specific Docker container image that can be used in various AWS services, such as Amazon SageMaker or Amazon Elastic Container Registry (Amazon ECR).

In [ ]:
%%capture
!pip3 install sagemaker==2.237.1

# INSTALL SAGEMAKER SDK FOR AWS DEPLOYMENT
# Why: Install the SageMaker Python SDK to enable programmatic deployment
# of the fine-tuned model to AWS SageMaker endpoints for production inference.
# Version 2.237.1 ensures compatibility with DJL deployment framework.

In [ ]:
import boto3
import json
import sagemaker.djl_inference
from sagemaker.session import Session
from sagemaker import image_uris
from sagemaker import Model
sagemaker_session = Session()
print("sagemaker_session: ", sagemaker_session)
aws_role = sagemaker_session.get_caller_identity_arn()
print("aws_role: ", aws_role)
aws_region = boto3.Session().region_name
print("aws_region: ", aws_region)
image_uri = image_uris.retrieve(framework="djl-deepspeed",
                                version="0.22.1",
                                region=sagemaker_session._region_name)
print("image_uri: ", image_uri)

# INITIALIZE AWS AND SAGEMAKER PARAMETERS FOR DEPLOYMENT
# Why: Set up AWS configuration required for model deployment:
# - sagemaker_session: Manages AWS credentials and API interactions
# - aws_role: IAM role ARN with permissions to access S3, SageMaker, ECR
# - aws_region: AWS region where resources will be deployed
# - image_uri: DJL DeepSpeed container image URI for efficient inference serving
# These parameters are prerequisites for creating and deploying endpoints.
# Retrieve the DJL DeepSpeed inference container image for the specified region


### <a name="step6.2">Step 6.2: Create the model artifact</a> ###

To upload the model artifact to the S3 bucket, we need to create a TAR GZ file that contains the model's parameters. First, create a directory named `lora_model` and a subdirectory named `dolly-3b-lora`. The "-p" option ensures that the command creates any intermediate directories if they don't exist. Then, copy the LoRA checkpoints, `adapter_model.bin` and `adapter_config.json`, to `dolly-3b-lora`. The base Dolly model is downloaded at runtime from the Hugging Face Hub.

In [ ]:
%%bash
rm -rf lora_model
mkdir -p lora_model
mkdir -p lora_model/dolly-3b-lora
cp dolly-3b-lora/adapter_config.json lora_model/dolly-3b-lora/
cp dolly-3b-lora/adapter_model.bin lora_model/dolly-3b-lora/

# CREATE MODEL ARTIFACT DIRECTORY STRUCTURE FOR DEPLOYMENT
# Why: Prepare a deployment-ready package containing:
# - adapter_config.json & adapter_model.bin: LoRA weights (loaded at runtime)
# - Base Dolly model: Downloaded from Hugging Face during deployment
# The directory structure matches DJL serving requirements for LoRA model serving.

Next, set the [DJL Serving configuration options](https://docs.aws.amazon.com/sagemaker/latest/dg/large-model-inference-configuration.html) in `serving.properties`. Using the Jupyter `%%writefile` magic command, you can write the following content to a file named lora_model/serving.properties.
- `engine=Python`: This line specifies the engine used for serving.
- `option.entryPoint=model.py`: This line specifies the entry point for the serving process, which is set to model.py.
- `option.adapter_checkpoint=dolly-3b-lora`: This line sets the checkpoint for the adapter to dolly-3b-lora. A checkpoint typically represents the saved state of a model or its parameters.
- `option.adapter_name=dolly-lora`: This line sets the name of the adapter to dolly-lora, a component that helps interface between the model and the serving infrastructure.

In [ ]:
%%writefile lora_model/serving.properties
engine=Python
option.entryPoint=model.py
option.adapter_checkpoint=dolly-3b-lora
option.adapter_name=dolly-lora

# CONFIGURE DJL SERVING PROPERTIES FOR LORA MODEL DEPLOYMENT
# Why: Define how DJL serving framework loads and serves the fine-tuned model:
# - engine=Python: Use Python runtime for DJL service
# - entryPoint=model.py: Custom Python script handling model loading and inference
# - adapter_checkpoint=dolly-3b-lora: Path to LoRA adapter weights
# - adapter_name=dolly-lora: Identifier for the LoRA adapter for the serving framework

You also need the environment requirement file in the model artifact. Create a file named `lora_model/requirements.txt` and write a list of Python package requirements, typically used with package managers such as `pip`.

In [ ]:
%%writefile lora_model/requirements.txt
accelerate>=0.16.0,<1
bitsandbytes==0.39.0
click>=8.0.4,<9
datasets>=2.10.0,<3
deepspeed>=0.8.3,<0.9
faiss-cpu==1.7.4
ipykernel==6.22.0
scipy==1.11.1
torch>=2.0.0
transformers==4.28.1
peft==0.3.0
pytest==7.3.2

# SPECIFY PYTHON DEPENDENCIES FOR DEPLOYMENT ENVIRONMENT
# Why: List all required Python packages for the inference container:
# - PyTorch & transformers: Model inference engine
# - accelerate: Multi-GPU/device support for serving
# - bitsandbytes: 8-bit quantization support (matches training setup)
# - peft: LoRA adapter loading and management
# - DeepSpeed: Optimized inference with model parallelization
# Versions must match or be compatible with training environment.

### <a name="step6.3">Step 6.3: Create the inference script</a>

Similar to the fine-tuning notebook, a custom pipeline, `InstructionTextGenerationPipeline`, is defined. The code is provided in `utils/deployment_model.py`.

You save these inference functions to `lora_model/model.py`.

In [ ]:
%%bash
cp utils/deployment_model.py lora_model/model.py

# COPY CUSTOM INFERENCE HANDLER TO MODEL ARTIFACT
# Why: Deploy the custom model.py inference script (from utils/deployment_model.py) which:
# - Loads the base Dolly model and LoRA adapter at inference time
# - Handles request preprocessing and tokenization
# - Performs model inference with loaded fine-tuned weights
# - Formats responses for end users

### <a name="step6.4">Step 6.4: Upload the model artifact to Amazon S3</a>

Create a compressed tarball archive of the lora_model directory and save it as lora_model.tar.gz.

In [ ]:
%%bash
tar -cvzf lora_model.tar.gz lora_model/

# CREATE COMPRESSED TARBALL OF MODEL ARTIFACT DIRECTORY
# Why: Package lora_model/ directory and all contents into a single compressed file
# for efficient S3 upload and SageMaker deployment:
# - Reduces upload size significantly
# - Required format for SageMaker model artifact versioning
# - -cvzf: Create (c), verbose (v), compressed gzip (z), from file (f)

Upload the lora_model.tar.gz file to the specified S3 bucket.

In [ ]:
import boto3
import json
import sagemaker.djl_inference
from sagemaker.session import Session
from sagemaker import image_uris
from sagemaker import Model
s3 = boto3.resource('s3')
s3_client = boto3.client('s3')
s3 = boto3.resource('s3')
for bucket in s3.buckets.all():
    if bucket.name.startswith('artifact'):  # Find bucket with 'artifact' prefix
        mybucket = bucket.name
        print(mybucket)
response = s3_client.upload_file("lora_model.tar.gz", mybucket, "lora_model.tar.gz")

# UPLOAD COMPRESSED MODEL ARTIFACT TO AWS S3 BUCKET
# Why: Store the model tarball in S3 (SageMaker's model repository):
# - S3 is required for SageMaker to access model data during deployment
# - boto3 S3 client enables programmatic file uploads
# - Model must be accessible by ARN (s3://bucket-name/key-path)
# This URL will be provided to SageMaker during endpoint creation.
# Locate the artifact bucket (AWS Lab environment has specific naming convention)
# Upload compressed model artifact to S3

### <a name="step6.5">Step 6.5: Deploy the model</a> ###

Now, it's the time to deploy the fine-tuned LLM by using the SageMaker Python SDK. The SageMaker Python SDK `Model` class is instantiated with the following parameters:

- `image_uri`: The Docker image URI that represents the deep learning framework and version to be used.
- `model_data`: The location of the fine-tuned LLM model artifact in an S3 bucket. It specifies the path to the TAR GZ file that contains the model's parameters, architecture, and any necessary artifacts.
- `predictor_cls`: This is just a JSON in JSON out predictor, nothing DJL related. For more information, see [sagemaker.djl_inference.DJLPredictor](https://sagemaker.readthedocs.io/en/stable/frameworks/djl/sagemaker.djl_inference.html#djlpredictor).
- `role`: The IAM role ARN that provides necessary permissions to access resources, such as the S3 bucket that contains the model data.

In [ ]:
model_data="s3://{}/lora_model.tar.gz".format(mybucket)
model = Model(image_uri=image_uri,
              model_data=model_data,
              predictor_cls=sagemaker.djl_inference.DJLPredictor,
              role=aws_role)

# CREATE SAGEMAKER MODEL OBJECT FOR ENDPOINT DEPLOYMENT
# Why: Instantiate a SageMaker Model class that specifies all deployment parameters:
# - image_uri: DJL DeepSpeed container for optimized inference serving
# - model_data: S3 path to the compressed model artifact (tar.gz)
# - predictor_cls: DJLPredictor handles JSON request/response serialization
# - role: IAM role grants SageMaker permissions to access S3 and other resources
# This Model object is then deployed to create an inference endpoint.

NOTE: The deployment should be completed within 10 minutes. Any longer than that, your endpoint might have failed.

In [ ]:
%%time
predictor = model.deploy(1, "ml.g4dn.2xlarge", wait=False)

# DEPLOY MODEL TO SAGEMAKER ENDPOINT WITH ASYNCHRONOUS INITIALIZATION
# Why: Create a production-ready inference endpoint:
# - 1 instance: Deploy one ml.g4dn.2xlarge GPU instance (NVIDIA T4 GPU)
# - ml.g4dn.2xlarge: Cost-effective GPU instance with 1x T4 GPU (16GB VRAM)
# - wait=False: Return immediately; deployment continues in background (takes 5-10 minutes)
# After deployment, endpoint accepts inference requests via predictor object.

In [ ]:
import time
sm_client = boto3.client('sagemaker')
while True:
    status = sm_client.describe_endpoint(EndpointName=predictor.endpoint_name)['EndpointStatus']
    print(f"Status: {status}")
    if status == 'InService':  # Endpoint is ready for inference
        break
    time.sleep(30)  # Wait 30 seconds before checking again

# POLL ENDPOINT STATUS UNTIL DEPLOYMENT COMPLETES
# Why: Monitor deployment progress and wait for endpoint to become operational:
# - Checks endpoint status every 30 seconds
# - 'InService' status means endpoint is ready to accept inference requests
# - Deployment typically takes 5-10 minutes (longer on first deployment)
# Prevents sending requests before endpoint is ready (would cause failures).

### <a name="step7">Step 7: Test the deployed inference</a>

Test the inference endpoint with [predictor.predict](https://sagemaker.readthedocs.io/en/stable/api/inference/predictors.html#sagemaker.predictor.Predictor.predict).

In [ ]:
outputs = predictor.predict({"inputs": "What security measures does Amazon SageMaker have?"})

# SEND INFERENCE REQUEST TO DEPLOYED ENDPOINT
# Why: Test the fine-tuned model with a real question about Amazon SageMaker:
# - predictor.predict(): Sends JSON request to the deployed endpoint
# - Input format: {"inputs": "Your question here"}
# - The fine-tuned model responds with relevant information from training data
# This demonstrates the model's ability to answer domain-specific queries after fine-tuning.

In [ ]:
from IPython.display import Markdown
Markdown(outputs)

# DISPLAY INFERENCE RESPONSE IN FORMATTED MARKDOWN
# Why: Render the model's response in readable markdown format for clean visualization.
# Shows the fine-tuned model's ability to generate coherent, relevant answers
# about the domain it was fine-tuned on (Amazon SageMaker FAQs).